In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_squared_error
from itertools import product

In [ ]:
df = pd.read_csv("tsla_2014_2023.csv")
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
df.head()

In [ ]:
# ── Use log-transformed close price for better stationarity & accuracy ──
data = np.log(df['close'])

train_size = int(len(data) * 0.8)
train = data[:train_size]
test  = data[train_size:]

print(f"Train size: {len(train)}, Test size: {len(test)}")

In [ ]:
# ── ADF test to confirm stationarity after differencing ──
result = adfuller(train.diff().dropna())
print(f"ADF Statistic : {result[0]:.4f}")
print(f"p-value       : {result[1]:.4f}")
print("Series is stationary" if result[1] < 0.05 else "Series is NOT stationary")

In [ ]:
# ── Grid-search for the best (p, d, q) order by AIC ──
# Searches p in [0..4], d in [1..2], q in [0..4]
best_aic   = np.inf
best_order = None

for p, d, q in product(range(5), range(1, 3), range(5)):
    try:
        aic = ARIMA(train, order=(p, d, q)).fit().aic
        if aic < best_aic:
            best_aic, best_order = aic, (p, d, q)
    except Exception:
        pass

print(f"Best ARIMA order: {best_order}  (AIC = {best_aic:.2f})")

In [ ]:
# ── Fit the best model ──
model_fit = ARIMA(train, order=best_order).fit()
print(model_fit.summary())

In [ ]:
# ── Rolling / walk-forward forecast (much more realistic & accurate) ──
# Instead of forecasting all test steps at once, we retrain on every new
# observation so the model always sees the latest data.

history     = list(train)
predictions_log = []

for t in range(len(test)):
    fit  = ARIMA(history, order=best_order).fit()
    yhat = fit.forecast(steps=1)[0]
    predictions_log.append(yhat)
    history.append(test.iloc[t])   # add the true value for the next step

# Convert back from log-space to original price space
predictions_log = np.array(predictions_log)
predictions = np.exp(predictions_log)
actual      = np.exp(test.values)

print("Rolling forecast complete.")

In [ ]:
# ── Evaluation ──
rmse     = np.sqrt(mean_squared_error(actual, predictions))
mae      = np.mean(np.abs(actual - predictions))
mape     = np.mean(np.abs((actual - predictions) / actual)) * 100
accuracy = 100 - mape          # accuracy based on MAPE (more reliable than RMSE-based)

print(f"RMSE            : {rmse:.4f}")
print(f"MAE             : {mae:.4f}")
print(f"MAPE            : {mape:.2f}%")
print(f"Approx Accuracy : {accuracy:.2f}%")

if accuracy >= 80:
    print("✅ Target accuracy of 80% achieved!")
else:
    print("⚠️  Still below 80% — consider more data or a hybrid model.")

In [ ]:
# ── Plot ──
plt.figure(figsize=(14, 6))
plt.plot(actual,      label='Actual Price',    linewidth=1.5)
plt.plot(predictions, label='Predicted Price', linewidth=1.5, linestyle='--')
plt.title(f'ARIMA{best_order} Rolling Forecast  |  Accuracy: {accuracy:.2f}%')
plt.xlabel('Time Step')
plt.ylabel('Price (USD)')
plt.legend()
plt.tight_layout()
plt.show()